In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from easyner.database.sqlite_backend.db_main import EasyNerDBHandler

db = EasyNerDBHandler()

In [ ]:
## Top 50 comparison Uniq Docs - Vs NPMI
# Plot two horizontal bar plots of the top 50 cooccurrences by fq_doc_level and npmi
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Get cooccurrences from database
cooccurrences_top_50_fq_doc_level = db.co.get_cooccurrences(min_pmi=0, min_uniq_docs=2, order_by="uniq_docs", limit=50)
cooccurrences_top_50_npmi = db.co.get_cooccurrences(max_npmi=1, min_uniq_docs=5, order_by="npmi", limit=50)

# Extract disease and phenomenon names from cooccurence.e1.txt and cooccurence.e2.txt
cooccurrences_top_50_fq_doc_level = [(co.e1.txt, co.e2.txt, co.fq_doc_level, co.uniq_docs) for co in cooccurrences_top_50_fq_doc_level]
cooccurrences_top_50_npmi = [(co.e1.txt, co.e2.txt, co.npmi, co.fq_doc_level, co.uniq_docs) for co in cooccurrences_top_50_npmi]

# Create dataframes
df_fq_doc_level = pd.DataFrame(cooccurrences_top_50_fq_doc_level, columns=["Disease", "Phenomenon", "FQ_DOC_LEVEL", "UNIQ_DOCS"])
df_npmi = pd.DataFrame(cooccurrences_top_50_npmi, columns=["Disease", "Phenomenon", "NPMI", "FQ_DOC_LEVEL", "UNIQ_DOCS"])
# Rename columns
df_npmi = df_npmi.rename(columns={"NPMI": "Normalized Pointwise Mutual Information", "FQ_DOC_LEVEL": "Freqency (document level)", "UNIQ_DOCS": "Unique Documents"})

# Change column names to be more descriptive
df_fq_doc_level = df_fq_doc_level.sort_values(by="UNIQ_DOCS", ascending=False)
df_fq_doc_level = df_fq_doc_level.rename(columns={"FQ_DOC_LEVEL": "Freqency (document level)", "UNIQ_DOCS": "Unique Documents"})

df_fq_doc_level


In [ ]:
from easyner.database.sqlite_backend.db_statistics.color_scheme import (
    NODE_COLOR_DIS_ONLY_RGB,
    NODE_COLOR_PNM_ONLY_RGB,
)
from easyner.database.sqlite_backend.db_statistics.cooccurence_horizontal_barplot import (
    CooccurenceHorizontalBarplot,
)

# Initialize the visualizer with appropriate colors
entity_pair_visualizer = CooccurenceHorizontalBarplot(
    first_entity_color=NODE_COLOR_DIS_ONLY_RGB,      # Light blue for diseases
    second_entity_color=NODE_COLOR_PNM_ONLY_RGB,     # Light green for phenomena
    separator='|',                      # Vertical line separator
    column_spacing=0.02,                # Minimal spacing between columns
    label_opacity=0.45,                  # Increased opacity for label backgrounds
    title_opacity=0.95,                  # Increased opacity for title backgrounds
)

# Create and show the NPMI plot
npmi_fig = entity_pair_visualizer.create_bar_plot(
    df=df_npmi,
    x_column="Normalized Pointwise Mutual Information",
)
plt.show()

# Create and show the Document Frequency plot
frequency_fig = entity_pair_visualizer.create_bar_plot(
    df=df_fq_doc_level,
    x_column="Unique Documents",
)
plt.show()